[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_71_Observability_Online_Evals.ipynb)

# Lesson 71 — Phase 8 Kickoff: Observability & Online Evals in Production

**Where you are:** 70 lessons done, across 7 phases. You have shipped **two** flagship
open-source tools — `paper-distiller` (Phase 6) and `agent-bench` (Phase 7) — each with a
CLI, a PyPI package, CI, and a launch. That is real, working, *installable* software.

So what is left? The thing nobody teaches until it bites them: **what happens after `pip install`,
when strangers run your agent on inputs you never imagined.**

That is Phase 8 — **Production Ops for LLM systems.** It is a different discipline from
everything so far. Until now you *built* things and evaluated them against suites *you wrote*
(`agent-bench`, offline, before deploy). Now we evaluate the system on **traffic you did not write** —
live production requests — and we do it *continuously*, not once before a release.

### Phase 8 roadmap (tentative & adaptive — not a contract)

| Lesson | Topic | New idea |
|---|---|---|
| **L71 (this one)** | **Observability & online evals** | traces, metrics, **evaluating live traffic** |
| L72 | Structured logging + trace search | querying traces, correlation IDs, sessions |
| L73 | Alerting, SLOs & on-call for LLM apps | error budgets, quality SLOs, paging |
| L74 | A/B testing & guarded rollouts | prompt/model experiments in prod, shadow traffic |
| L75 | Feedback loops & data flywheel | thumbs-up/down → eval set → next release |
| L76 | Phase 8 capstone: a prod-obs layer for `agent-bench` | ship it as a real module |

> This is genuinely *new* — Lesson 48 covered OpenTelemetry tracing mechanics and Lesson 49
> covered running a large **offline** eval. This phase is about the **online** world: sampling and
> scoring *production* traffic, and watching for **drift** away from your offline baseline.


## 1. The core distinction: offline eval vs online eval

You already own an *offline* evaluator — `agent-bench`. Internalize exactly how the online world differs,
because almost every production incident lives in the gap between these two columns.

| | **Offline eval (`agent-bench`, L61–70)** | **Online eval (this lesson)** |
|---|---|---|
| **Inputs** | a fixed task suite *you wrote* | live user traffic *you can't enumerate* |
| **When** | once, before deploy (gates CI) | continuously, on live requests |
| **Ground truth** | you have labels / expected answers | usually **no** labels — you infer quality |
| **Scoring** | can be exact-match, deterministic | heuristics + **LLM-judge** + user feedback |
| **Coverage** | only what you thought to test | the long tail of real inputs |
| **Failure it catches** | regressions vs known-good cases | **drift**, novel inputs, silent quality decay |
| **Cost model** | pay once per eval run | must run *cheaply* on a *sample* of prod traffic |

**Neither replaces the other.** Offline eval is your pre-flight checklist; online eval is your
cockpit instruments *during the flight*. A model that passes every offline test can still degrade in
production because real inputs drift away from your suite. You only see that if you are watching.

### The three pillars of LLM observability

1. **Traces** — a structured record of *one request's* journey through your agent (the LLM calls, tool
   calls, retrievals), with timing, token usage, and cost per step. This is the *unit of debugging*.
2. **Metrics** — aggregates over many traces as time-series: cost/day, p95 latency, error rate,
   tokens/request. This is the *unit of alerting*.
3. **Online evals** — a *quality* signal computed on a **sample** of live traces (heuristics, an
   LLM-judge, or human feedback). This is the *unit of trust*: "is the thing still good?"

We build a small, vendor-neutral version of all three, then wire it onto a `paper-distiller`-shaped
agent — with **zero** external services and **no API key required**.


In [ ]:
# === Setup — runs on Colab or any Python 3.9+; NO API KEY REQUIRED ===
# The whole lesson runs on a deterministic offline simulation of production traffic.
# If you DO set an ANTHROPIC_API_KEY (Colab: key icon -> ANTHROPIC_API_KEY), the
# LLM-judge online scorer will additionally make one real call so you can see it live.
import sys, subprocess
def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
_pip("rich", "pydantic>=2", "matplotlib", "pandas", "numpy", "nest_asyncio", "anthropic")

import os, json, time, uuid, random, statistics, asyncio, contextlib, math
from dataclasses import dataclass, field, asdict
from typing import Optional, Any
import numpy as np
import nest_asyncio; nest_asyncio.apply()   # lets asyncio.run work inside a notebook (L61+ guard)
from rich.console import Console
from rich.table import Table
# force_jupyter=False keeps Rich from bypassing stdout capture — the L64 pitfall, reused.
console = Console(force_jupyter=False, no_color=False, highlight=False)

# Where the notebook writes its module files. On Colab this is /content.
CONTENT = "/content"
os.makedirs(CONTENT, exist_ok=True)

# Graceful-degrade API key check (never crash if it is absent).
try:
    from google.colab import userdata            # type: ignore
    API_KEY = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    API_KEY = os.environ.get("ANTHROPIC_API_KEY")
HAVE_API_KEY = bool(API_KEY)
random.seed(71); np.random.seed(71)             # deterministic "production" for reproducibility
print("Setup OK.  HAVE_API_KEY =", HAVE_API_KEY, "(lesson runs fully either way)")


## 2. Pillar 1 — Tracing: a span tree over the agent loop

A **span** is one timed unit of work: "the LLM call", "the retrieval", "the whole request".
Spans nest — a request span contains an LLM span and a tool span — forming a tree.

We follow the industry's **OpenTelemetry GenAI semantic conventions** for the *names* of the
attributes we record (`gen_ai.usage.input_tokens`, `gen_ai.system`, ...). That matters: if you name
attributes the standard way, you can later export to *any* backend (Langfuse, Phoenix, Honeycomb,
Datadog) without re-instrumenting. We are building a tiny local tracer, but with **portable field names**.

Below, `Tracer` is a context-manager span recorder that:
- assigns each span an id and links it to its parent (so we get the tree),
- records start/end time → **latency**,
- lets you attach attributes (tokens, cost, model, status),
- appends every finished span to a **JSONL sink** (one span per line) — exactly how real tracing
  backends ingest data, and trivial to `grep`.


In [ ]:
# === A minimal, portable tracer (context-manager spans -> JSONL sink) ===
TRACE_LOG = os.path.join(CONTENT, "traces.jsonl")
open(TRACE_LOG, "w").close()  # truncate at start of run

@dataclass
class Span:
    name: str
    trace_id: str
    span_id: str
    parent_id: Optional[str]
    start: float
    end: Optional[float] = None
    status: str = "ok"                 # "ok" | "error"
    attrs: dict = field(default_factory=dict)
    @property
    def latency_ms(self) -> float:
        return round(((self.end or self.start) - self.start) * 1000, 2)

class Tracer:
    """Vendor-neutral span recorder. One JSONL line per finished span."""
    def __init__(self, sink_path: str):
        self.sink_path = sink_path
        self._stack: list[Span] = []          # currently-open spans (for parent linking)

    @contextlib.contextmanager
    def span(self, name: str, trace_id: Optional[str] = None, **attrs):
        parent = self._stack[-1] if self._stack else None
        s = Span(
            name=name,
            trace_id=trace_id or (parent.trace_id if parent else uuid.uuid4().hex[:12]),
            span_id=uuid.uuid4().hex[:12],
            parent_id=parent.span_id if parent else None,
            start=time.perf_counter(),
            attrs=dict(attrs),
        )
        self._stack.append(s)
        try:
            yield s                            # caller mutates s.attrs / s.status inside the block
        except Exception as e:
            s.status = "error"
            s.attrs["error.type"] = type(e).__name__
            s.attrs["error.message"] = str(e)[:200]
            raise
        finally:
            s.end = time.perf_counter()
            self._stack.pop()
            with open(self.sink_path, "a") as f:
                rec = asdict(s); rec["latency_ms"] = s.latency_ms
                f.write(json.dumps(rec) + "\n")

tracer = Tracer(TRACE_LOG)

# Quick self-test: a nested span pair should produce a parent/child link.
with tracer.span("demo.request") as root:
    root.attrs["gen_ai.system"] = "anthropic"
    with tracer.span("demo.llm_call") as child:
        time.sleep(0.01)
        child.attrs["gen_ai.usage.input_tokens"] = 42
assert child.parent_id == root.span_id and child.trace_id == root.trace_id
print("Tracer self-test passed: child linked to parent, shared trace_id.")
open(TRACE_LOG, "w").close()  # clear the demo span before real traffic


## 3. A `paper-distiller`-shaped agent, instrumented — and simulated production traffic

We do not need the real network or a real API key to learn observability. We need a *stream of
requests that behaves like production*: mostly-fine, occasionally slow, occasionally failing, with
realistic token counts and cost. So we build `distill_one()` — the same shape as the real
`paper-distiller.distill()` — wrapped in tracer spans, and drive it with a **traffic generator**.

Two things we deliberately bake into the simulation so the rest of the lesson has something to *find*:
- a small baseline **error rate** (timeouts / parse failures), and
- a block of traffic partway through where inputs drift **out of distribution** (papers from a domain
  the agent handles poorly) — this is the **drift** our online eval must catch.

Cost is computed from tokens using a simple public-style price table (input/output per-1K-token rates).


In [ ]:
# === The instrumented agent + a production-traffic generator ===
PRICE = {"in_per_1k": 0.003, "out_per_1k": 0.015}   # illustrative $ / 1K tokens
def cost_usd(tin, tout):
    return round(tin/1000*PRICE["in_per_1k"] + tout/1000*PRICE["out_per_1k"], 6)

# Two synthetic "input distributions". In-domain papers distill well; the drift block
# feeds papers the agent summarizes poorly (short, generic, low-quality outputs).
GOOD_TITLES = ["Attention Is All You Need", "Deep Residual Learning", "BERT Pretraining",
               "Denoising Diffusion Models", "LoRA Low-Rank Adaptation"]
DRIFT_TITLES = ["Untitled draft v3", "misc notes (scanned)", "??? preprint", "table of contents only"]

def _mock_distill_output(title, drifted):
    # Deterministic stand-in for a Claude call. Good inputs -> rich summary; drifted -> thin/garbled.
    if drifted:
        return {"summary": "This paper is about stuff.", "key_points": [], "n_tokens_out": random.randint(20, 45)}
    return {"summary": f"'{title}' proposes a method with clear contributions, experiments, and results.",
            "key_points": ["problem", "method", "experiments", "results"],
            "n_tokens_out": random.randint(180, 320)}

def distill_one(paper_id: str, drifted: bool = False) -> dict:
    """One production request, fully traced. Returns the response dict (incl. its trace_id)."""
    title = random.choice(DRIFT_TITLES if drifted else GOOD_TITLES)
    with tracer.span("distill.request", **{"gen_ai.system": "anthropic",
                                           "input.paper_id": paper_id,
                                           "input.title": title}) as root:
        # 1) fetch span
        with tracer.span("distill.fetch") as fs:
            time.sleep(random.uniform(0.002, 0.006)); fs.attrs["http.status"] = 200
        # 2) the LLM span — the expensive one. Inject rare failures + occasional latency spikes.
        with tracer.span("distill.llm_call") as ls:
            fail = random.random() < 0.04                 # 4% baseline error rate
            slow = random.random() < 0.08                 # 8% latency spike
            time.sleep(random.uniform(0.05, 0.09) + (0.15 if slow else 0.0))
            if fail:
                raise TimeoutError("upstream model call timed out")
            tin = random.randint(1500, 3000)
            out = _mock_distill_output(title, drifted)
            tout = out["n_tokens_out"]
            ls.attrs.update({"gen_ai.request.model": "claude-sonnet-mock",
                             "gen_ai.usage.input_tokens": tin,
                             "gen_ai.usage.output_tokens": tout,
                             "gen_ai.cost_usd": cost_usd(tin, tout)})
            root.attrs.update({"output.summary": out["summary"],
                               "output.n_key_points": len(out["key_points"]),
                               "gen_ai.cost_usd": cost_usd(tin, tout)})
        return {"trace_id": root.trace_id, "title": title, **out}

def generate_traffic(n=200, drift_from=140, drift_to=175):
    """Simulate n production requests. Requests in [drift_from, drift_to) are out-of-distribution."""
    responses = []
    for i in range(n):
        drifted = drift_from <= i < drift_to
        try:
            responses.append(distill_one(f"paper-{i:04d}", drifted=drifted))
        except TimeoutError:
            responses.append({"trace_id": None, "title": None, "error": "timeout"})
    return responses

responses = generate_traffic()
n_ok = sum(1 for r in responses if "error" not in r)
print(f"Generated {len(responses)} production requests -> {n_ok} ok, {len(responses)-n_ok} errored.")
print("Trace log lines (spans):", sum(1 for _ in open(TRACE_LOG)))


## 4. Pillar 2 — Metrics: aggregate the trace log into a dashboard

Traces are per-request; **metrics** are what you put on a wall and alert on. We read the JSONL sink
back and compute the numbers an on-call engineer actually watches:

- **error rate** (fraction of request spans with `status == "error"`),
- **latency** p50 / p95 / p99 of the request span (tails matter more than the mean — one user's p99
  is a rage-quit),
- **cost**: total $ and $ per request,
- **throughput** and **tokens** per request.

Notice we compute these from the *same* trace data — no separate metrics pipeline. That is the point of
structured traces: metrics are just aggregations over them.


In [ ]:
# === Roll the JSONL trace log up into request-level metrics ===
def load_spans(path):
    return [json.loads(l) for l in open(path) if l.strip()]

spans = load_spans(TRACE_LOG)
req_spans = [s for s in spans if s["name"] == "distill.request"]
llm_spans = [s for s in spans if s["name"] == "distill.llm_call"]

def pct(vals, p):
    if not vals: return 0.0
    vals = sorted(vals); k = (len(vals)-1) * p/100
    lo, hi = math.floor(k), math.ceil(k)
    if lo == hi: return vals[int(k)]
    return vals[lo] + (vals[hi]-vals[lo]) * (k-lo)

lat = [s["latency_ms"] for s in req_spans]
errs = sum(1 for s in req_spans if s["status"] == "error")
# error spans are the llm span; the request span propagates status via the tracer's except-block
err_req = sum(1 for s in req_spans if s["status"] == "error")
costs = [s["attrs"].get("gen_ai.cost_usd", 0.0) for s in llm_spans]
tin = [s["attrs"].get("gen_ai.usage.input_tokens", 0) for s in llm_spans]
tout = [s["attrs"].get("gen_ai.usage.output_tokens", 0) for s in llm_spans]

metrics = {
    "requests":        len(req_spans),
    "error_rate":      round(err_req/len(req_spans), 4) if req_spans else 0.0,
    "latency_p50_ms":  round(pct(lat, 50), 1),
    "latency_p95_ms":  round(pct(lat, 95), 1),
    "latency_p99_ms":  round(pct(lat, 99), 1),
    "cost_total_usd":  round(sum(costs), 4),
    "cost_per_req_usd": round(sum(costs)/max(1,len(costs)), 6),
    "avg_input_tokens":  int(statistics.mean(tin)) if tin else 0,
    "avg_output_tokens": int(statistics.mean(tout)) if tout else 0,
}

t = Table(title="Production metrics dashboard (from trace log)")
t.add_column("metric", style="cyan"); t.add_column("value", justify="right", style="green")
for k, v in metrics.items():
    t.add_row(k, str(v))
console.print(t)


## 5. Pillar 3 — Online evals: score a *sample* of live traffic

Here is the hard part. Offline, you had expected answers. Online, a user asks the agent to distill a
paper and **nobody knows the "right" summary** — there is no label. So how do you know quality is holding?

Three signals, cheapest first:
1. **Heuristics** — cheap, deterministic proxies (Did we return key points? Is the summary suspiciously
   short? Did output tokens collapse?). Catch gross failures for free, on 100% of traffic.
2. **LLM-as-judge** — a *second* model call that rates the output 1–5 against a rubric. Higher signal,
   but it **costs money and latency**, so you run it on a **sample**, *asynchronously, off the hot path*
   (never make the user wait for the judge).
3. **Human feedback** — thumbs up/down (Lesson 75). Highest signal, lowest volume.

Two rules that separate a real system from a toy:
- **Sample.** Judging 100% of traffic doubles your bill. A few percent, uniformly sampled, is enough to
  track a distribution.
- **Off the hot path.** The judge runs in the background; the user already got their answer.

Below: a heuristic scorer on every response, and an **async** LLM-judge on a sample. The judge has an
offline deterministic fallback so this cell runs with no API key — and makes one real call if you have one.


In [ ]:
# === Online evaluation: heuristic on all, async LLM-judge on a sample ===
def heuristic_score(resp: dict) -> dict:
    """Cheap, label-free quality proxies. Returns a 0..1 score + reasons. Runs on 100% of traffic."""
    if "error" in resp:
        return {"score": 0.0, "reasons": ["request_errored"]}
    reasons = []; score = 1.0
    if resp.get("n_tokens_out", 0) < 60:          # summary suspiciously short
        score -= 0.5; reasons.append("output_too_short")
    if not resp.get("key_points"):                # no structured key points extracted
        score -= 0.4; reasons.append("no_key_points")
    return {"score": round(max(0.0, score), 3), "reasons": reasons or ["ok"]}

async def llm_judge(resp: dict) -> float:
    """Rate summary quality 1..5 (normalized to 0..1). Real call if HAVE_API_KEY, else offline proxy."""
    if "error" in resp:
        return 0.0
    if HAVE_API_KEY:
        try:
            import anthropic
            client = anthropic.Anthropic(api_key=API_KEY)
            prompt = ("Rate this paper summary's quality 1-5 (5=excellent). "
                      "Reply with ONLY the digit.\n\nSUMMARY:\n" + resp.get("summary", ""))
            msg = client.messages.create(model="claude-sonnet-4-20250514", max_tokens=5,
                                          messages=[{"role": "user", "content": prompt}])
            digit = "".join(c for c in msg.content[0].text if c.isdigit())[:1] or "3"
            return int(digit) / 5.0
        except Exception:
            pass  # fall through to offline proxy on any API error
    # Offline deterministic proxy: judge correlates with the heuristic signal + a little noise.
    base = heuristic_score(resp)["score"]
    await asyncio.sleep(0.001)                      # simulate the judge's async latency
    return round(min(1.0, max(0.0, base * 0.9 + random.uniform(-0.05, 0.05))), 3)

def sample_indices(n, rate=0.30, seed=71):
    rng = random.Random(seed)
    return [i for i in range(n) if rng.random() < rate]

async def run_online_evals(responses, sample_rate=0.30):
    # Heuristic on ALL traffic (cheap):
    for r in responses:
        r["heuristic"] = heuristic_score(r)
    # LLM-judge on a SAMPLE only, concurrently, off the hot path:
    idx = sample_indices(len(responses), sample_rate)
    judged = await asyncio.gather(*[llm_judge(responses[i]) for i in idx])
    for i, j in zip(idx, judged):
        responses[i]["judge"] = j
    return idx

sampled = asyncio.run(run_online_evals(responses))
heur_mean = statistics.mean(r["heuristic"]["score"] for r in responses)
judge_mean = statistics.mean(responses[i]["judge"] for i in sampled)
print(f"Heuristic scored 100% of traffic ({len(responses)} reqs), mean = {heur_mean:.3f}")
print(f"LLM-judge scored a {len(sampled)/len(responses):.0%} sample ({len(sampled)} reqs), mean = {judge_mean:.3f}")
print("Judge mode:", "REAL Claude calls" if HAVE_API_KEY else "offline deterministic proxy")


## 6. Drift detection: compare production quality against the offline baseline

Now the payoff. `agent-bench` gave us an **offline baseline** quality score before deploy. Online, we
compute a **rolling** quality score over a sliding window of recent requests. When the rolling score
falls a meaningful amount **below the offline baseline**, that is **drift** — the system is quietly
getting worse on real traffic, even though nothing in the code changed.

This is the single most valuable thing observability buys you: catching decay *before your users churn*,
not after. We deliberately injected a drift block into the traffic (requests 140–175). A good monitor
should light up there and **nowhere else**.

We use a simple, explainable rule (rolling mean over a window vs. an absolute threshold). Production
systems add statistical tests (e.g. a two-sample test on score distributions, or PSI on input features),
but the *mental model* is exactly this: **baseline vs. rolling, alarm on the gap.**


In [ ]:
# === Rolling quality vs offline baseline -> drift alarm ===
OFFLINE_BASELINE = 0.92    # pretend agent-bench reported this pre-deploy (heuristic-comparable scale)
WINDOW = 20                # rolling window size (requests)
ALARM_DROP = 0.20          # alarm if rolling mean falls this far below baseline

series = [r["heuristic"]["score"] for r in responses]
rolling, alarms = [], []
for i in range(len(series)):
    w = series[max(0, i-WINDOW+1): i+1]
    rm = statistics.mean(w)
    rolling.append(rm)
    if rm < OFFLINE_BASELINE - ALARM_DROP:
        alarms.append(i)

# Group contiguous alarm indices into human-readable incident ranges.
incidents = []
for i in alarms:
    if incidents and i == incidents[-1][1] + 1:
        incidents[-1][1] = i
    else:
        incidents.append([i, i])

print(f"Offline baseline quality: {OFFLINE_BASELINE}")
print(f"Rolling window: {WINDOW} reqs | alarm threshold: < {OFFLINE_BASELINE-ALARM_DROP:.2f}")
if incidents:
    for a, b in incidents:
        lo = min(rolling[a:b+1])
        print(f"  DRIFT ALARM: requests {a}-{b}  (rolling quality dipped to {lo:.2f})")
else:
    print("  No drift detected.")

# The injected drift was requests 140-175. Assert the monitor fires inside that block and is
# not screaming across the whole run (i.e. it is specific, not a stuck alarm).
assert any(140 <= a <= 175 or 140 <= b <= 175 for a, b in incidents), "monitor missed the injected drift!"
assert len(alarms) < len(series) * 0.5, "monitor is over-firing (not specific)"
print("\nDrift monitor validated: fired on the injected out-of-distribution block, stayed quiet otherwise.")


## 7. The dashboard chart — cost, tail latency, and quality over time

One picture an on-call engineer can read in two seconds: three stacked time-series. Watch how the
**quality** panel craters over the drift block while cost and latency look *fine* — which is exactly
why quality needs its **own** signal. Latency and cost dashboards alone would have shown you nothing.


In [ ]:
# === Three-panel observability chart -> saved PNG ===
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Per-request cost & latency aligned to request order (skip errored requests for cost).
order_cost, order_lat = [], []
li = 0
for s in [s for s in spans if s["name"] == "distill.request"]:
    order_lat.append(s["latency_ms"])
for s in [s for s in spans if s["name"] == "distill.llm_call"]:
    order_cost.append(s["attrs"].get("gen_ai.cost_usd", 0.0))

fig, ax = plt.subplots(3, 1, figsize=(10, 8), sharex=False)
ax[0].plot(order_cost, lw=1); ax[0].set_title("Cost per request ($)"); ax[0].set_ylabel("$")
ax[1].plot(order_lat, lw=1, color="tab:orange"); ax[1].set_title("Request latency (ms)"); ax[1].set_ylabel("ms")
ax[2].plot(rolling, lw=1.5, color="tab:green"); ax[2].axhline(OFFLINE_BASELINE, ls="--", color="gray", label="offline baseline")
ax[2].axhline(OFFLINE_BASELINE-ALARM_DROP, ls=":", color="red", label="alarm threshold")
ax[2].axvspan(140, 175, color="red", alpha=0.12, label="injected drift")
ax[2].set_title("Rolling quality (online eval)"); ax[2].set_ylabel("score"); ax[2].set_xlabel("request #"); ax[2].legend(loc="lower left", fontsize=8)
plt.tight_layout()
CHART = os.path.join(CONTENT, "obs_dashboard.png")
plt.savefig(CHART, dpi=110); plt.close()
print("Saved dashboard chart ->", CHART)
print("Read it top-to-bottom: cost & latency look healthy through the drift block; only the")
print("quality panel (bottom) reveals the incident. That gap is why online evals exist.")


## 8. Wire it into a reusable module (the Phase-8 way)

Same discipline as every prior phase: the concepts become a small module you can import. We write
`observability/tracing.py` (the `Tracer`/`Span`) and `observability/online_eval.py` (heuristic +
sampler + rolling drift monitor) to disk, so `agent-bench` and `paper-distiller` can both depend on
them. Lesson 76 turns this into a shipped, tested module; today we lay the file structure.


In [ ]:
# === Write the observability module to disk ===
import textwrap
pkg = os.path.join(CONTENT, "observability")
os.makedirs(pkg, exist_ok=True)

tracing_py = textwrap.dedent(
    '''
    # observability/tracing.py — portable span tracer (OTel GenAI-style attribute names).
    import time, uuid, json, contextlib
    from dataclasses import dataclass, field, asdict
    from typing import Optional

    @dataclass
    class Span:
        name: str; trace_id: str; span_id: str; parent_id: Optional[str]
        start: float; end: Optional[float] = None; status: str = "ok"
        attrs: dict = field(default_factory=dict)
        @property
        def latency_ms(self): return round(((self.end or self.start) - self.start) * 1000, 2)

    class Tracer:
        # One JSONL line per finished span; nests via an internal stack for parent linking.
        def __init__(self, sink_path): self.sink_path = sink_path; self._stack = []
        @contextlib.contextmanager
        def span(self, name, trace_id=None, **attrs):
            parent = self._stack[-1] if self._stack else None
            s = Span(name, trace_id or (parent.trace_id if parent else uuid.uuid4().hex[:12]),
                     uuid.uuid4().hex[:12], parent.span_id if parent else None,
                     time.perf_counter(), attrs=dict(attrs))
            self._stack.append(s)
            try:
                yield s
            except Exception as e:
                s.status = "error"; s.attrs["error.type"] = type(e).__name__; raise
            finally:
                s.end = time.perf_counter(); self._stack.pop()
                rec = asdict(s); rec["latency_ms"] = s.latency_ms
                with open(self.sink_path, "a") as f: f.write(json.dumps(rec) + "\\n")
    '''
).strip() + "\n"

online_eval_py = textwrap.dedent(
    '''
    # observability/online_eval.py — label-free online quality signals + drift monitor.
    import statistics

    def heuristic_score(resp: dict) -> dict:
        # Cheap proxies you can run on 100% of production traffic (no labels needed).
        if "error" in resp: return {"score": 0.0, "reasons": ["request_errored"]}
        score, reasons = 1.0, []
        if resp.get("n_tokens_out", 0) < 60: score -= 0.5; reasons.append("output_too_short")
        if not resp.get("key_points"):       score -= 0.4; reasons.append("no_key_points")
        return {"score": round(max(0.0, score), 3), "reasons": reasons or ["ok"]}

    def sample_indices(n, rate=0.30, seed=71):
        import random; rng = random.Random(seed)
        return [i for i in range(n) if rng.random() < rate]

    class DriftMonitor:
        # Rolling mean of a quality signal vs a fixed offline baseline; alarm on the gap.
        def __init__(self, baseline, window=20, alarm_drop=0.20):
            self.baseline, self.window, self.alarm_drop = baseline, window, alarm_drop
            self.buf = []
        def update(self, score) -> bool:
            self.buf.append(score); self.buf = self.buf[-self.window:]
            return statistics.mean(self.buf) < self.baseline - self.alarm_drop
    '''
).strip() + "\n"

init_py = "from .tracing import Tracer, Span\nfrom .online_eval import heuristic_score, sample_indices, DriftMonitor\n"

for fname, content in [("tracing.py", tracing_py), ("online_eval.py", online_eval_py), ("__init__.py", init_py)]:
    with open(os.path.join(pkg, fname), "w") as f:
        f.write(content)

# Import the freshly written module and smoke-test the DriftMonitor end to end.
import importlib.util
def _load(mod, path):
    spec = importlib.util.spec_from_file_location(mod, path); m = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(m); return m
oe = _load("online_eval_mod", os.path.join(pkg, "online_eval.py"))
mon = oe.DriftMonitor(baseline=0.92, window=5, alarm_drop=0.2)
fired = [mon.update(s) for s in [0.95, 0.93, 0.4, 0.3, 0.35, 0.9]]
assert any(fired), "DriftMonitor failed to fire on a low-quality run"
print("Wrote observability/ module:", sorted(os.listdir(pkg)))
print("DriftMonitor smoke-test fired correctly on injected low scores.")


## 9. Pitfalls (the ones that cause real 3 a.m. pages)

| # | Pitfall | Why it bites |
|---|---|---|
| 1 | **Watching only cost & latency, not quality** | the drift block above was invisible on both — quality needs its *own* signal |
| 2 | **LLM-judge on 100% of traffic** | doubles your bill and adds latency; sample instead |
| 3 | **Judge on the hot path** | the user waits for a second model call they never asked for; always run it async/off-path |
| 4 | **No baseline to compare against** | "quality is 0.7" is meaningless without the offline number it should be near |
| 5 | **Mean latency instead of p95/p99** | the mean hides the tail; your angriest users live at p99 |
| 6 | **Logging full prompts/outputs with PII** | traces are a data-retention & privacy liability — redact before you sink |
| 7 | **Non-standard attribute names** | naming tokens `toks` instead of `gen_ai.usage.input_tokens` locks you out of every OTel backend |
| 8 | **Sampling bias** | sampling only fast/cheap requests hides exactly the slow, expensive, failing ones you need to see |
| 9 | **Trusting the judge blindly** | LLM-judges drift and have their own biases; periodically calibrate them against human labels |
| 10 | **Alarms with no hysteresis** | a monitor that flips on every single dip pages you into ignoring it — use a rolling window |


In [ ]:
# === Verification checklist (assertions, not vibes) ===
checks = []
def check(name, cond): checks.append((name, bool(cond))); return bool(cond)

check("trace log written",                os.path.exists(TRACE_LOG) and os.path.getsize(TRACE_LOG) > 0)
check("spans nest (parent links exist)",  any(s["parent_id"] for s in spans))
check("GenAI attr names present",         any("gen_ai.usage.input_tokens" in s["attrs"] for s in spans))
check("metrics computed",                 metrics["requests"] > 0 and metrics["latency_p95_ms"] >= metrics["latency_p50_ms"])
check("error rate in plausible range",    0.0 <= metrics["error_rate"] <= 0.15)
check("heuristic scored 100% of traffic", all("heuristic" in r for r in responses))
check("judge scored only a sample",       0 < len(sampled) < len(responses))
check("drift alarm fired in 140-175",     any(140 <= a <= 175 or 140 <= b <= 175 for a, b in incidents))
check("drift monitor is specific",        len(alarms) < len(series) * 0.5)
check("dashboard chart saved",            os.path.exists(CHART))
check("observability module written",     all(os.path.exists(os.path.join(pkg, f)) for f in ["tracing.py","online_eval.py","__init__.py"]))

t = Table(title="Lesson 71 verification")
t.add_column("check", style="cyan"); t.add_column("result", justify="right")
for name, ok in checks:
    t.add_row(name, "[green]PASS[/green]" if ok else "[red]FAIL[/red]")
console.print(t)
passed = sum(1 for _, ok in checks if ok)
assert passed == len(checks), f"{len(checks)-passed} checks failed"
print(f"All {passed}/{len(checks)} checks passed.")


## 10. Summary, homework & what's next

### What you learned

| Concept | One-line takeaway |
|---|---|
| Offline vs online eval | offline = fixed suite pre-deploy; online = live traffic, continuously, usually unlabeled |
| Three pillars | **traces** (debug unit), **metrics** (alert unit), **online evals** (trust unit) |
| Portable tracing | context-manager spans → JSONL, named with OTel GenAI conventions so any backend can ingest |
| Metrics from traces | error rate, p50/p95/p99 latency, cost/req, tokens — all *aggregations over spans* |
| Online scoring | heuristics on 100%, LLM-judge on a **sample**, **off the hot path** |
| Drift detection | rolling quality vs offline baseline; alarm on the gap — the thing cost/latency dashboards miss |

### Homework

1. **Redaction:** add a `redact()` step to the tracer that strips emails/keys from `attrs` before writing
   the JSONL line (pitfall #6). Prove it with a trace containing a fake `sk-ant-...` string.
2. **Real judge calibration:** if you have an API key, run `llm_judge` for real on 20 responses, hand-label
   the same 20 yourself, and compute the correlation. How much do you trust the judge?
3. **Two-sample drift test:** replace the absolute-threshold alarm with a statistical test (e.g. compare
   the score distribution of the last 30 requests vs the first 30 with `scipy.stats.mannwhitneyu`).
4. **Cost guardrail:** add a rolling **cost-per-request** alarm alongside the quality alarm and trip it by
   inflating output tokens in the traffic generator.
5. **Wire it live:** import `observability/` into your real `agent-bench` and emit one span per benchmark
   task run — your offline harness now produces online-shaped traces.

### Next lesson (L72) — Structured logging & trace search
Today we *produced* traces. Next we make them *queryable*: correlation IDs, session grouping, and a tiny
trace-search layer so you can answer "show me every failed request from user X in the last hour" — the
day-to-day of actually operating an LLM service.

> Phase 8 so far: L71 observability & online evals ✓. Six phases and ~71 lessons in, you can now *build*,
> *ship*, **and operate** an LLM system. That last verb is what separates a demo from a product.
